# Gateway Verifikasi Dokumen Akademik AI (Google Colab)

Notebook ini adalah versi Colab dari `main.py`.

Urutan pakai:
1. Jalankan cell install dependencies.
2. **Mount Google Drive.**
3. Upload model `model_verifikasi_lstm.keras`.
4. Isi `SUPABASE_URL` dan `SUPABASE_API_KEY`.
5. Jalankan cell API setup.
6. (Opsional) Jalankan server publik via ngrok dan tes endpoint
- Isi ngrok.set_auth_token menggunakan token di https://dashboard.ngrok.com/get-started/your-authtoken
- Jalankan sel server ngrok

In [ ]:
!pip -q install fastapi uvicorn python-multipart tensorflow supabase python-dotenv requests pyngrok nest-asyncio

In [ ]:
import os
import hashlib
import numpy as np
from fastapi import FastAPI, File, UploadFile, HTTPException, Form
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from supabase import create_client, Client

MODEL_PATH = '/content/model_verifikasi_lstm.keras'
MAX_LEN = 10000
model = None
supabase = None

def init_model(model_path: str = MODEL_PATH):
    global model
    if not os.path.exists(model_path):
        print(f'WARNING: Model belum ditemukan di {model_path}. Upload dulu, lalu jalankan init_model().')
        model = None
        return None
    model = load_model(model_path)
    print('Model berhasil dimuat.')
    return model

def init_supabase():
    global supabase
    supabase_url = os.getenv('SUPABASE_URL')
    supabase_api_key = os.getenv('SUPABASE_API_KEY')
    if not supabase_url or not supabase_api_key:
        print('WARNING: SUPABASE_URL / SUPABASE_API_KEY belum diisi. Data history tidak akan tersimpan.')
        supabase = None
        return None
    supabase = create_client(supabase_url, supabase_api_key)
    print('Supabase client berhasil dibuat.')
    return supabase

In [ ]:
from google.colab import files

uploaded = files.upload()

if 'model_verifikasi_lstm.keras' in uploaded:
    with open(MODEL_PATH, 'wb') as f:
        f.write(uploaded['model_verifikasi_lstm.keras'])
    print(f'Model disimpan ke {MODEL_PATH}')
else:
    print('Jika nama file model berbeda, ubah MODEL_PATH atau rename file saat upload.')

In [ ]:
os.environ['SUPABASE_URL'] = 'https://your-supabase-url.supabase.co'
os.environ['SUPABASE_API_KEY'] = 'your-supabase-api-key'

In [ ]:
app = FastAPI(title='Gateway Verifikasi Dokumen Akademik AI (Colab)')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

@app.post('/api/verify/')
async def verify_document(file: UploadFile = File(...), user_id: str = Form(...)):
    if file.content_type not in ['application/pdf', 'application/octet-stream']:
        raise HTTPException(status_code=400, detail='File harus berupa PDF.')

    if not file.filename.lower().endswith('.pdf'):
        raise HTTPException(status_code=400, detail='Kesalahan Ekstensi: Format file harus .pdf')

    if model is None:
        raise HTTPException(status_code=500, detail='Model verifikasi belum dimuat. Jalankan init_model() dulu.')

    try:
        file_bytes = await file.read()
        file_hash = hashlib.sha256(file_bytes).hexdigest()

        byte_array = list(file_bytes)
        if len(byte_array) > MAX_LEN:
            half_len = MAX_LEN // 2
            processed_bytes = byte_array[:half_len] + byte_array[-half_len:]
        else:
            processed_bytes = byte_array

        input_data = pad_sequences([processed_bytes], maxlen=MAX_LEN, padding='pre', truncating='pre')
        prediction = model.predict(input_data, verbose=0)

        probability_score = float(prediction[0][0])

        if probability_score > 0.5:
            status = 'PALSU'
        elif probability_score < 0.5:
            status = 'ASLI'
        else:
            status = 'PERLU_REVIEW'

        prediction_data = {
            'user_id': user_id,
            'file_name': file.filename,
            'file_type': file.content_type,
            'persentase': probability_score,
            'ai_classification': status
        }

        if supabase is not None:
            supabase.table('history').insert(prediction_data).execute()

        return JSONResponse(content={
            'nama_file': file.filename,
            'hash_sha256': file_hash,
            'status_verifikasi': status,
            'akurasi_prediksi': probability_score
        })
    except Exception as e:
        raise HTTPException(status_code=500, detail=f'Terjadi kesalahan saat memproses file: {str(e)}')

@app.get('/')
async def root():
    return {'message': 'Selamat datang di Gateway Verifikasi Dokumen Akademik AI. Gunakan endpoint /api/verify/ untuk memverifikasi dokumen PDF.'}

init_model()
init_supabase()

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok
import threading # Import threading

nest_asyncio.apply()

# set token dari https://dashboard.ngrok.com/get-started/your-authtoken)
ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN")

# Jalankan server di background lalu expose ke internet via ngrok
public_url = ngrok.connect(8000).public_url
print('Public URL:', public_url)

# Jalankan server Uvicorn di thread terpisah.
config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level="info")
server = uvicorn.Server(config)

# Jalankan server di thread terpisah agar tidak memblokir event loop utama Colab
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

In [ ]:
# Contoh test endpoint dari notebook
import requests

BASE_URL = 'http://127.0.0.1:8000'
PDF_PATH = '/content/ORI_001.pdf'  # ganti dengan path PDF yang ingin diuji
USER_ID = '11fd1f04-d484-44e7-b7b9-8c909a1aee28' # user: ilham

with open(PDF_PATH, 'rb') as f:
    files_payload = {'file': ('contoh.pdf', f, 'application/pdf')}
    data_payload = {'user_id': USER_ID}
    r = requests.post(f'{BASE_URL}/api/verify/', files=files_payload, data=data_payload)

print('status_code:', r.status_code)
print('response:', r.json())